# Теория ЛР 05: Drift Monitoring и Retraining Policy

## Перед началом
- **Для кого эта ЛР**: для студента, который начинает с нуля и пока не уверен в статистике и ML-терминах.
- **Зачем это в реальной жизни**: модель в продакшене стареет, даже если код не менялся, потому что меняются данные и поведение пользователей.
- **Что получится в конце**: вы сможете доказательно объяснить, почему в конкретном окне выбрано `observe` или `retrain`, с опорой на поля четырех CSV.
- **Что можно не знать заранее**: строгие математические доказательства тестов и архитектуру промышленного MLOps.
- **Что делать, если застрял**: вернитесь к карточкам терминов ниже, затем ответьте на блок "Проверь себя" в текущем разделе.

## Как читать этот notebook, если вы совсем новичок
1. Сначала читаете блоки **на пальцах**.
2. Потом смотрите **официальный термин** (латиница).
3. Затем связываете термин с **конкретной колонкой CSV**.
4. Только после этого переходите к формуле.

## Карманный словарь латиницы (первое знакомство)

### Карточка 1. Какие бывают сдвиги данных
- **Что это на пальцах**: мы сравниваем "вчерашний мир" данных с "сегодняшним" и смотрим, что именно изменилось.
- **Официальный термин: `covariate`**.
- **Официальный термин: `prior`**.
- **Официальный термин: `combined`**.
- **Зачем это в ЛР05**: тип сдвига помогает понять, насколько рискованно оставлять текущую модель без переобучения.
- **Где видно в артефактах**: `drift_detection_audit.csv`, колонка `scenario`.
- **Типичная ошибка новичка**: считать, что любой сдвиг автоматически означает `retrain`.
- **Короткий контрпример**: слабый `covariate`-сдвиг есть, но `delta_cost_vs_reference` почти не растет, значит можно оставить `observe`.

### Карточка 2. База статистического решения
- **Что это на пальцах**: тест — это не "магия", а формальная проверка, не похоже ли отличие на случайность.
- **Официальный термин: `H0`**.
- **Официальный термин: `H1`**.
- **Официальный термин: `KS`**.
- **Официальный термин: `chi2`**.
- **Официальный термин: `p-value`**.
- **Официальный термин: `alpha`**.
- **Официальный термин: `power`**.
- **Официальный термин: `PSI`**.
- **Зачем это в ЛР05**: мы не хотим ни ложных тревог, ни пропуска реального ухудшения.
- **Где видно в артефактах**: `drift_detection_audit.csv`, колонки `detector`, `p_value`, `effect_size`, `drift_flag`.
- **Типичная ошибка новичка**: принимать решение по одному числу `p_value`.
- **Короткий контрпример**: `p_value` маленький, но `PSI` маленький и цена ошибки не растет — немедленный `retrain` не обязателен.

### Карточка 3. Качество, калибровка и стоимость
- **Что это на пальцах**: модель может угадывать класс приемлемо, но давать слишком дорогие ошибки.
- **Официальный термин: `confusion matrix`**.
- **Официальный термин: `precision`**.
- **Официальный термин: `recall`**.
- **Официальный термин: `f1`**.
- **Официальный термин: `threshold`**.
- **Официальный термин: `Brier`**.
- **Официальный термин: `ECE`**.
- **Официальный термин: `expected_cost`**.
- **Зачем это в ЛР05**: policy использует не абстрактное качество, а полезность решения в практическом смысле.
- **Где видно в артефактах**: `monitoring_quality_audit.csv` (`f1`, `brier`, `ece`, `expected_cost`, `delta_*`) и `retraining_policy_decisions.csv`.
- **Типичная ошибка новичка**: смотреть только на `accuracy` и не видеть структуру ошибок.
- **Короткий контрпример**: `accuracy` почти не изменилась, но `expected_cost` вырос заметно — решение может быть `retrain`.

### Карточка 4. Экспериментальная гигиена
- **Что это на пальцах**: если эксперимент нечестный, любые красивые метрики бесполезны.
- **Официальный термин: `random_state`**.
- **Официальный термин: `stratify`**.
- **Официальный термин: `data leakage`**.
- **Зачем это в ЛР05**: нам нужно, чтобы выводы можно было повторить и проверить.
- **Где видно в артефактах**: косвенно во всех CSV через стабильность и правдоподобность метрик.
- **Типичная ошибка новичка**: принять "слишком идеальные" метрики за успех, не проверив утечку.
- **Короткий контрпример**: после утечки все метрики резко улучшились, но на реальных окнах качество падает.


## Раздел 0. Почему эта тема важна именно новичку
### Идея
- Главная мысль: модель может деградировать без единого изменения в коде.
- Причина: меняются входные данные, доля классов и структура ошибок.
- Поэтому нужен понятный ритуал: **заметили сигнал -> оценили риск -> приняли действие**.

### Формула
- Рабочая цепочка ЛР05:
  - `изменение данных` ->
  - `статистические сигналы` ->
  - `метрики качества и стоимости` ->
  - `policy-решение`.

### Мини-пример
- В марте в банк приходили клиенты с одним профилем дохода, в апреле — с другим.
- Модель начинает чаще ошибаться в дорогих случаях (`FN`), хотя внешне "все примерно нормально".
- Без мониторинга команда узнает о проблеме слишком поздно.

### Как читать результат/график
- Сначала ответьте простыми словами:
  - что поменялось в данных;
  - как изменились метрики;
  - какое действие теперь безопаснее.

### Если объяснять человеку без техбэкграунда
- "Мы делаем регулярный техосмотр модели: измеряем показатели и принимаем понятное решение — наблюдать или чинить".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1, шаг 1 (подготовка входа) и шаг 2 (первые сигналы).


## Раздел 1. Карта финальной темы
### Идея
- Здесь важна не одна формула, а **последовательность**.
- Если перескочить шаг, итоговое решение станет недоказуемым.

### Формула
- Маршрут ЛР05:
  - `reference` ->
  - мониторинговые окна ->
  - `drift_detection_audit.csv` ->
  - `monitoring_quality_audit.csv` ->
  - `retraining_policy_decisions.csv` ->
  - `post_retrain_comparison.csv`.

### Мини-пример
- Если окно `combined` дает рост доли drift-признаков и рост стоимости, policy фиксирует `retrain`.
- Если сдвиг слабый и стоимость стабильна, policy оставляет `observe`.

### Как читать результат/график
- Читайте ноутбук как конвейер: результат шага N является входом шага N+1.
- Любой вывод обязан ссылаться на конкретную таблицу и колонку.

### Если объяснять человеку без техбэкграунда
- "Сначала измеряем температуру, потом ставим диагноз, и только затем выбираем лечение".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1 (шаги 2-3), практика 2 (шаги 2-3).


## Раздел 2. Типы drift: covariate, prior, combined
### Идея
- На пальцах у нас три сценария:
  - изменились признаки;
  - изменилась доля класса;
  - изменились и признаки, и доля класса.

### Формула
- сдвиг профиля признаков (`covariate`): `P_t(X) != P_ref(X)`;
- сдвиг доли класса (`prior`): `P_t(Y) != P_ref(Y)`;
- комбинированный сдвиг (`combined`): оба условия верны одновременно.

### Мини-пример
- Кредитный кейс:
  - `covariate`: изменились возраст/доход клиентов;
  - `prior`: изменилась доля дефолтов;
  - `combined`: и профиль, и доля дефолтов изменились вместе.

### Как читать результат/график
- Сначала смотрим `scenario` в `drift_detection_audit.csv`.
- Потом сравниваем, где выше `drift_flag` и где растет `delta_cost_vs_reference`.

### Если объяснять человеку без техбэкграунда
- "В магазин пришли другие покупатели, и одновременно изменилась доля возвратов товара".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1, шаг 2 (`scenario`) и шаг 3 (сценарная сводка).


## Раздел 3. KS и chi2 как статистические детекторы
### Идея
- Тест помогает отделить устойчивый сигнал от случайного шума.
- Для числовых признаков используем `KS`, для категориальных — `chi2`.

### Формула
- Проверка гипотез:
  - `H0`: отличия нет;
  - `H1`: отличие есть;
  - если `p-value < alpha`, отличие статистически заметно.
- В ЛР05 фиксировано: `alpha = 0.05`.

### Мини-пример
- `income`: `p-value = 0.01` -> отличие статистически заметно.
- `segment`: `p-value = 0.40` -> тест не подтвердил отличие.

### Как читать результат/график
- Последовательность чтения:
  1. Какой тест сработал (`detector`).
  2. Насколько убедителен сигнал (`p_value`).
  3. Насколько сильный эффект (`effect_size = PSI`).

#### Нулевой статистический модуль для новичка
- **Генеральная совокупность**: весь воображаемый мир объектов (например, все будущие заявки).
- **Выборка**: ограниченный срез данных, с которым мы реально работаем.
- **Частота и доля**: сколько раз событие встретилось и какую часть это составляет.
- **Квантиль**: граница, делящая значения на равные доли (например, медиана — 50%).
- **Распределение**: как часто встречаются разные значения признака.
- **Ошибка I рода**: ложная тревога (сдвига нет, а тест сказал "есть").
- **Ошибка II рода**: пропуск проблемы (сдвиг есть, а тест сказал "нет").
- **Мощность (`power`)**: способность теста увидеть реальное отличие.

#### Когда НЕ верить тесту без допроверки
- Слишком маленький размер окна.
- Очень редкие категории в `chi2`.
- Есть признаки `data leakage` или несопоставимые условия сравнения.

### Если объяснять человеку без техбэкграунда
- "Тест — как датчик дыма: он полезен, но иногда дает ложную тревогу, поэтому проверяем контекст".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1, шаг 2 (`detector`, `p_value`).


## Раздел 4. PSI как мера силы сдвига
### Идея
- `p-value` отвечает на вопрос "заметно ли отличие статистически".
- `PSI` отвечает на вопрос "насколько отличие большое на практике".
- Поэтому эти сигналы читаются только в паре.

### Формула
- `PSI = sum((p_i - q_i) * ln(p_i / q_i))`.
- В ЛР05 фиксированы ориентиры:
  - `psi_warn = 0.10`;
  - `psi_alert = 0.25`.

### Мини-пример
- `p-value = 0.003`, `PSI = 0.04` -> сигнал статистический, но эффект слабый.
- `p-value = 0.003`, `PSI = 0.31` -> и заметно, и сильно, риск высокий.

### Как читать результат/график
- Практический алгоритм:
  1. Проверяем `p_value`.
  2. Смотрим `effect_size` (`PSI`).
  3. Подтверждаем по `delta_f1_vs_reference` и `delta_cost_vs_reference`.

#### Карточка типичной ошибки
- **Что это на пальцах**: высокий `PSI` — это сильный аргумент, но не автоматический приговор.
- **Зачем это в ЛР05**: чтобы не делать лишний `retrain` и не тратить ресурсы зря.
- **Где видно в артефактах**: `drift_detection_audit.csv`, поля `effect_size` и `drift_flag`.
- **Типичная ошибка новичка**: смотреть только на `drift_flag`, не проверяя влияние на стоимость.
- **Короткий контрпример**: `drift_flag=True`, но `delta_cost_vs_reference` остается около нуля.

### Если объяснять человеку без техбэкграунда
- "`p-value` говорит: изменение не похоже на случайность. `PSI` говорит: насколько это изменение вообще большое".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1, шаг 2 и шаг 3.


## Раздел 5. Связь drift с качеством и стоимостью
### Идея
- Drift сам по себе не равен бизнес-риску.
- Нужна связка: структура ошибок + качество + цена ошибок.

### Формула
- Из `confusion matrix` получаем:
  - `precision = TP / (TP + FP)`;
  - `recall = TP / (TP + FN)`;
  - `f1` как баланс `precision` и `recall`.
- Дополнительно:
  - `threshold` меняет баланс ошибок;
  - `Brier` и `ECE` показывают честность вероятностей;
  - `expected_cost` переводит ошибки в практический ущерб.

### Мини-пример
- `f1` упал умеренно, но `expected_cost` вырос сильно.
- Для policy это часто важнее, чем косметические колебания вспомогательных метрик.

### Как читать результат/график
- Порядок чтения:
  1. Есть ли деградация `f1`.
  2. Растет ли `expected_cost`.
  3. Подтверждается ли это drift-сигналами.
  4. Какое действие оправдано.

### Если объяснять человеку без техбэкграунда
- "Нам важно не только сколько раз модель угадала, но и сколько стоят ее ошибки".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1, шаг 3 и практика 2, шаг 2.


## Раздел 6. Policy-правило решения observe/retrain
### Идея
- Policy нужна, чтобы два разных аналитика приняли одинаковое решение на одних и тех же данных.

### Формула
- `retrain`, если выполняется хотя бы одно:
  - `drift_feature_share >= 0.30`, или
  - `delta_f1_vs_reference <= -0.05`, или
  - `delta_cost_vs_reference >= +0.15`.
- Иначе: `observe`.

### Мини-пример
- `drift_feature_share=0.12`, `delta_f1=-0.01`, `delta_cost=+0.18` -> `retrain` по стоимости.

### Как читать результат/график
- Всегда читаем пару: `policy_action` + `trigger_reason`.
- Без `trigger_reason` решение нельзя считать объяснимым.

### Если объяснять человеку без техбэкграунда
- "Это как медицинский протокол: если выполнен хотя бы один красный критерий, действуем сразу".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 2, шаг 2.


## Раздел 7. Что меняет retrain и как это проверять
### Идея
- Переобучение — это не цель, а гипотеза: "после retrain станет лучше".
- Гипотезу нужно проверить на числах.

### Формула
- `gain_f1 = f1_after - f1_before`.
- `gain_cost = expected_cost_before - expected_cost_after`.

### Мини-пример
- До retrain: `f1=0.61`, `expected_cost=0.34`.
- После retrain: `f1=0.68`, `expected_cost=0.24`.

### Как читать результат/график
- Сравниваем только сопоставимые пары `before_retrain`/`after_retrain` внутри одного сценария.
- Успех: `f1` растет и/или `expected_cost` снижается.

### Если объяснять человеку без техбэкграунда
- "Если ремонт не улучшил работу машины, значит ремонт был неудачным".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 2, шаг 3.


## Раздел 8. Ограничения и риски
### Идея
- Слишком частый `retrain` может быть дорогим и нестабильным.
- Слишком редкий `retrain` накапливает ошибки и убытки.

### Формула
- Компромисс: минимизировать одновременно
  - пропуск реального ухудшения,
  - и лишние перезапуски переобучения.

### Мини-пример
- Если запускать retrain на каждый слабый сигнал, вы получите "дрожание" системы без явной пользы.

### Как читать результат/график
- Смотрите динамику по окнам, а не один случай.
- Разовое улучшение не гарантирует устойчивый эффект.

### Если объяснять человеку без техбэкграунда
- "Нельзя чинить двигатель каждый день и нельзя вообще не чинить — нужен разумный график".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 2, шаг 4 (финальная рекомендация).


## Раздел 9. Чеклист интерпретации мониторинга
### Идея
- Чтобы новичок не терялся, используем повторяемый чеклист.

### Формула
- Для каждого окна проходите один и тот же порядок:
  1. `drift_feature_share`.
  2. `delta_f1_vs_reference`.
  3. `delta_cost_vs_reference`.
  4. `policy_action` и `trigger_reason`.

### Мини-пример
- Даже низкий `drift_feature_share` не гарантирует `observe`, если `delta_cost_vs_reference` уже высокий.

### Как читать результат/график
- Решение считается доказанным, если каждая фраза связана с полем CSV.
- Формат хорошего вывода: **тезис -> поле -> действие**.

### Если объяснять человеку без техбэкграунда
- "Это чек-лист врача: каждый пункт обязателен, иначе диагноз ненадежный".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 1, шаг 3 и практика 2, шаг 2.


## Раздел 10. Проверь себя
### Идея
- Самопроверка нужна, чтобы уметь объяснить решение другому человеку, а не просто повторить формулу.

### Формула
- **Ожидаемый формат ответа на каждый вопрос**:
  - 1 тезис простыми словами;
  - 1 ссылка на поле CSV;
  - 1 итоговое действие (`observe` или `retrain`).

### Мини-пример
- "Одного `p-value` мало: я проверяю еще `effect_size`, `delta_f1_vs_reference` и `delta_cost_vs_reference`, после чего выбираю действие".

### Как читать результат/график
- Если ответ нельзя привязать к колонке CSV, он пока неполный.

### Если объяснять человеку без техбэкграунда
- "Я не просто говорю мнение — я показываю, в какой строке таблицы это видно".

### Где это в практическом ноутбуке
- **Где это применится через 5 минут**: практика 2, шаги 2-4.

Вопросы для самопроверки:
1. Чем отличается статистическая заметность (`p-value`) от практической силы эффекта (`PSI`)?
2. В каком случае `retrain` оправдан даже при небольшом падении `f1`?
3. Как объяснить нетехническому руководителю риск слишком частого переобучения?
4. Какие поля из четырех CSV вы покажете, чтобы защитить итоговое решение по сценарию?
5. Почему без `random_state`, `stratify` и контроля `data leakage` выводам нельзя доверять?
